# Setup

In [ ]:
import sys, json
from pathlib import Path
from openai import OpenAI
import pandas as pd
from pprint import pprint
from gitsource import chunk_documents, GithubRepositoryDataReader
from tqdm.auto import tqdm
import numpy as np
from minsearch import VectorSearch, Index


sys.path.append(Path().absolute().parent._str)
from src.embed.embedder import Embedder
from src.evaluation_utils import (
    llm_structured_retry, Questions
)

In [ ]:
openai_client = OpenAI()

- This homework continues from homework 2. We reuse the same chunks and the same search functions, so it's easiest to keep working in the same project.
- Load the data exactly as in homework 2. This gives 72 pages.

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(len(documents))

# Generating ground truth

- To evaluate search, we need a dataset of questions where we know which document is the correct answer. This is the ground truth.
- We generate it the same way as in the module.
  - For each lesson page, we ask an LLM to write 5 questions that are answered by that page.
  - Each question is then labeled with the page it came from.
- We use the same structured-output approach as in the module - the same `Questions` model and the `llm_structured` helper from `evaluation_utils.py`.
- The module's instructions generate questions from a FAQ record, so we adapt them for a lesson page.
- We ask for different wording from the page on purpose. Real users don't phrase their questions the way the lesson does, and copying the text would make the evaluation too easy.

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

- For each page, build a JSON user prompt from its `filename` and `content`
- then call `llm_structured` with the `Questions` model.
- Turn each returned question into a record labeled with the page's `filename`.
- The call also returns the token usage, the same as in the lessons.

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

# Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

In [ ]:
sample_fns = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]
sample_docs = [doc for doc in documents if doc['filename'] in sample_fns]

ground_truth = []
usages = []

for doc in sample_docs:
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

Each call returns the token usage, which most LLM APIs report on the response object (e.g. `response.usage.input_tokens` / `prompt_tokens`).
What's the average number of input tokens across these 3 calls?

> These numbers vary between runs, even with the same model, so pick the closest
> option. A different provider or model may land further apart, but the input
> tokens stay in the same order of magnitude - the prompt we send is the same.

In [ ]:
print(sum([usage.input_tokens for usage in usages]) / len(usages))

* 140
* 1400
* 14000
* 140000

# The full ground truth

- You don't need to generate the data for the rest of the homework.
- We already did it for all 72 pages, using the same approach as in the lessons, and saved the 360 questions to a file.
- Download it

In [ ]:
# %%bash
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv -O hw-ground-truth.csv -q

Load it with pandas into a list of records called `ground_truth`. Each record
has a `question` and the `filename` of the page that should answer it.

In [ ]:
gt_df = pd.read_csv("hw-ground-truth.csv")
ground_truth = gt_df.to_dict(orient="records")
print(len(ground_truth))
pprint(ground_truth[0])

# Searching the chunks

- We search over the same chunks as in homework 2.
- Create them with `chunk_documents`.
- This gives 295 chunks.

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))
print(chunks[0].keys())

- Now rebuild the search from homework 2 over these chunks.
    - Build a text index (`Index`) and a vector index (`VectorSearch`), both keyed on `filename`.
    - Wrap each one in a function, `text_search` and `vector_search`, that takes a query and the number of results to return (5 by default).
- For hybrid search, reuse the `rrf` function from homework 2.
- Then define `hybrid_search` on top of it.

In [ ]:
# %%bash
# uv run python ../src/embed/download.py

In [ ]:
embed = Embedder(path="models/Xenova/all-MiniLM-L6-v2")
texts = [chunk['content'] for chunk in chunks]
batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)
X = np.array(X)

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks)

In [ ]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(chunks)

In [ ]:
def text_search(q, k):
    return index.search(q, num_results=k)

def vec_search(q, k):
    return vindex.search(embed.encode(q), num_results=k)

def rrf(result_lists, k, rff_k):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (rff_k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:k]]

def hybrid_search(q, k, rff_k):
    text_results = text_search(q, k)
    vector_results = vec_search(q, k)
    return rrf([text_results, vector_results], k, rff_k)

# Q2. First result with text search

Take the first question from the ground truth

In [ ]:
q = ground_truth[0]["question"]

After running `text_search` for it, what's the `filename` of the first result?

In [ ]:
res = text_search(q, 5)
print(res[0]['filename'])

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/10-rag-next-steps.md`

# Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of the first result?

In [ ]:
res = vec_search(q, 5)
print(res[0]['filename'])

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `04-evaluation/lessons/11-evaluation-intro.md`
* `04-evaluation/lessons/12-rag-answers.md`

- This question was generated from `01-agentic-rag/lessons/01-intro.md`.
- Notice that one method finds the right page at the top and the other doesn't.
- That's exactly why we measure across the whole dataset instead of trusting one query.

## Evaluation metrics

- We evaluate search exactly as in the module, reusing the same functions from the lecture.
    - We change only the label. Our ground truth uses `filename`
    - so a result counts as a hit when a returned chunk's `filename` matches the question's `filename`, not a document `id`.

# Q4. Evaluating text search

- Evaluate `text_search` on the ground truth data.

In [ ]:
def compute_relevance(q, search_function, k):
    doc_id = q["filename"]
    results = search_function(q=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function, k=5):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, k=k)
        relevance_total.append(relevance)

    return relevance_total


def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
relevance_score_all = compute_relevance_total(ground_truth, text_search)

In [ ]:
print(hit_rate(relevance_score_all))

What's the Hit Rate?
* 0.55
* 0.66
* 0.76
* 0.88

# Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the module only evaluated keyword search.

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [ ]:
relevance_score_all_vec = compute_relevance_total(ground_truth, vec_search)
print(mrr(relevance_score_all_vec))

What's the MRR?

* 0.35
* 0.45
* 0.55
* 0.65

# Q6. Tuning hybrid search

- The `k` constant in RRF controls how much the top ranks matter. A smaller `k` sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data
- so let's measure it.
- Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1, 50, 100, and 200. Compare the MRR values for these runs.

In [ ]:
from functools import partial
relevance_dict = dict()
for rff_k in [1,50, 100, 200]:
    relevance_dict[rff_k] = compute_relevance_total(
        ground_truth,
        partial(hybrid_search, rff_k=rff_k)
    )

In [ ]:
for (k,v) in relevance_dict.items():
    print(f"rff_k: {k} | mrr: {mrr(v)}")

Which `k` gives the best MRR?
* 1
* 50
* 100
* 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.

# Using this framework

- You now have an `evaluate` function that takes any search function and returns Hit Rate and MRR.
- Use it to measure any change you make to search:
    - tune the field boosts in keyword search
    - try a different embedding model for vector search
    - change `k` in the RRF formula for hybrid search
    - change the number of results you return
- Change a setting, re-run `evaluate`, and see whether the metric moves. The ground truth stays fixed, so the comparison is fair. That's how you replace guessing with measuring.